## Task 1: Multi-Object Based Synopsis
- background modeling
- object detection
- euclidean distance tracker
- 

In [3]:
# task 1: multi-object base synopsis

# imports:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
#background modeling:

frame_count = 0

# function for extracting frames from the videos and returns the grey frames in an array
def extract_frames(video_path):
  '''takes the filepath of the video and extracts its frames. Returns an array of grey frames '''
  cap = cv2.VideoCapture(video_path)

  # Check if video opened successfully
  if not cap.isOpened():
    print("Error opening video file")

  # get video details for output
  frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = cap.get(cv2.CAP_PROP_FPS)
  frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  print(f"frame size: {frame_width}x{frame_height}")
  print(f"fps:{fps}")
  print(f"frame count:{frame_count}")

  count = 0
  frames = [] # initialize array
  # open the video and loop thru the frames to extract them
  print("looping thru frames to extract:")
  while(cap.isOpened()):
    # print("inside loop")
    ret, frame = cap.read() # read the frame
    if ret == True:
      print(f"extracting frame number {count}")
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # make sure only one grey channel instead of rgb
      frames.append(frame) # save the frame into list
      count += 1
    else:
          print("End of video or error occurred.")
          break
  cap.release()
  return frames

# function to get BG0 given the frames
def get_BG0(frames, step=5):
  ''' median of every step-th frame, so moving objects disappear '''
  sample = np.stack(frames[::step], axis=0)
  return np.median(sample, axis=0).astype(np.uint8)


# FGit = It ? |BGit+1 - It| > T : 0 OR FGit = It ? |BGit - It| > T : 0
# function to calculate FG from curr frame and next BG OR curr BG
def foreground(It, BGt_1):
  '''takes the curr frame and the curr background model. Returns the foreground (FG) and the mask(Msk)'''
  rows, cols = It.shape
  FG = np.zeros((rows,cols), dtype = np.uint8) # create empty foreground
  Msk = np.zeros((rows,cols), dtype = np.uint8) # create empty Mask
  thresh = 30 # edited
  # for i in range(rows):
  #   for j in range(cols):
  #     diff = BGt_1[i,j].astype(np.int16) - It[i,j].astype(np.int16)
  #     diff = np.abs(diff) # get abs diff betw next background and curr frame
  #     if diff > thresh: # check diff with threshold
  #       FG[i,j] = It[i,j]
  #       Msk[i,j] = 255
  #     else:
  #       FG[i,j] = 0
  #       Msk[i,j] = 0
  diff = np.abs(BGt_1.astype(np.int16)-It.astype(np.int16))
  FG = np.where((diff > thresh), It, 0).astype(np.uint8)
  Msk = np.where((diff > thresh), 255, 0).astype(np.uint8)
  return FG, Msk

# function to update background model -  need to use Msk instead of beta
def background_beta(It, BGt, beta):
  '''takes the curr frame, the curr background model, and beta. Calculates next background model '''
  rows, cols = It.shape
  BGt_1 = np.zeros((rows,cols), dtype = np.uint8) # create empty next BG model
  # for i in range(rows):
  #   for j in range(cols):
  #     w_sum = (1-beta) * BGt[i,j].astype(np.int16) + beta * It[i,j].astype(np.int16) # calculate weighted sum using beta
  #     BGt_1[i,j] = round(w_sum) # convert to int using roundign and put into next BG model
  w_sum = (1- beta) * BGt.astype(np.float32) + beta * It.astype(np.float32)
  BGt_1 = np.round(w_sum).astype(np.uint8)
  return BGt_1

# alternative, uses the background model creator that doesn't use masks
# function to create the foreground images, the corresponding mask images, and updates the background model over all frames
def moving_average_BG_alternate(frames, BGt, beta, path):
  ''' takes the frames, beta, and BG0 (BGt) to get the foreground, mask, and updated background models for each frame'''

  c = len(frames)
  for i in range(1,c): # loop over frames starting from second frame
    frame = frames[i] # get the curr frame
    # BGt_1 = background(frame, BGt, beta) # next background model
    # CHECK THIS!!!!!
    # FG, Msk = foreground(frame, BGt_1) # foreground using next background
    FG, Msk = foreground(frame, BGt) # foreground using curr background
    BGt_1 = background_beta(frame, BGt, beta) # next background model


    # write as images
    cv2.imwrite(f"{path}/Background/background_alternate_{i:04}.png",BGt_1)
    cv2.imwrite(f"{path}/Foreground/foreground_alternate_{i:04}.png",FG)
    cv2.imwrite(f"{path}/Mask/mask_alternate_{i:04}.png",Msk)
    BGt = BGt_1.copy() # set curr BG model for next iteration as the next background model from this iteration

def writeVideo(output_path, frames_path, fps=30):
    ''' used to create output videos from the frames'''  
    # create codec for writing the video
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    # create output for writing the grey video
    cap = cv2.VideoCapture(frames_path) # sequence of images

    if not cap.isOpened():
        print(f"Error: Could not open image sequence at {frames_path}")
        return

    # 2. Get dimensions automatically from the first image frame
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"frame size: {frame_width}x{frame_height}")
    print(f"fps:{fps}")
    print(f"frame count:{frame_count}")

    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height), isColor=True)


    count = 0
    # loop through the image sequence and write into the out vid
    print("looping through image sequence to write vid")
    while(cap.isOpened()):
        # print("inside writing video loop")
        ret, frame = cap.read()
        if ret == True:
            out.write(frame)
            print(f"writing frame number {count}")
            count += 1
        else:
                print("End of video or error occurred.")
                break

    cap.release()
    out.release()

beta = 0.02 # edited

# test on video
frames1 = extract_frames('tabletennis_short.mp4')

BG0_1 = get_BG0(frames1) # prev - BGt
cv2.imwrite(f"Results/Background/background_alternate_0000.png",BG0_1)

# using only beta for background model
moving_average_BG_alternate(frames1, BG0_1, beta, 'Results' )
writeVideo('tabletennis_foregrounds_alternate.mp4', 'Results/Foreground/foreground_alternate_%04d.png')
writeVideo('tabletennis_backgrounds_alternate.mp4', 'Results/Background/background_alternate_%04d.png')
writeVideo('tabletennis_masks_alternate.mp4', 'Results/Mask/mask_alternate_%04d.png')
print("saved videos")



# object detection: background subtraction + morphology + connected components

def clean_mask(Msk, open_k=3, close_k=21, dilate_iter=3):
    '''takes the raw mask from foreground() and cleans it with morphology'''
    Msk = Msk.astype(np.uint8)
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    k_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    Msk = cv2.morphologyEx(Msk, cv2.MORPH_OPEN, k_small)    # remove specks
    Msk = cv2.morphologyEx(Msk, cv2.MORPH_CLOSE, k_big)     # fill holes
    Msk = cv2.dilate(Msk, k_small, iterations=dilate_iter)  # merge fragments
    return Msk

def merge_boxes(dets, gap=25):
    '''merges boxes that overlap or are within `gap` pixels of each other'''
    boxes = [[x, y, x+w, y+h] for (x, y, w, h, cx, cy) in dets]
    merged = True
    while merged:
        merged = False
        out = []
        while boxes:
            a = boxes.pop()
            i = 0
            while i < len(boxes):
                b = boxes[i]
                if (a[0] - gap < b[2] and b[0] - gap < a[2] and
                    a[1] - gap < b[3] and b[1] - gap < a[3]):
                    a = [min(a[0], b[0]), min(a[1], b[1]),
                         max(a[2], b[2]), max(a[3], b[3])]
                    boxes.pop(i)
                    merged = True
                else:
                    i += 1
            out.append(a)
        boxes = out
    return [(x1, y1, x2-x1, y2-y1, (x1+x2)//2, (y1+y2)//2) for x1, y1, x2, y2 in boxes]

def detect_objects(frame, BG, min_area=1500):
    '''takes a grey frame and the background model.
    Returns a list of (x, y, w, h, cx, cy) per blob, and the cleaned mask'''
    _, Msk = foreground(frame, BG)
    Msk = clean_mask(Msk)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(Msk, connectivity=8)
    detections = []
    for i in range(1, n):  # label 0 is the background
        x, y, w, h, area = stats[i]
        if area >= min_area:
            cx, cy = centroids[i]
            detections.append((int(x), int(y), int(w), int(h), int(cx), int(cy)))
    return merge_boxes(detections), Msk   # <- changed: merge before returning


# euclidean distance tracker and synopsis generation


# Create the Synopsis folder (exist_ok=True means it won't error if it's already there)
os.makedirs('Results/Synopsis', exist_ok=True)

class Track:
    """One tracked object: id, last centre, last frame seen, trajectory, crop, and initial bbox."""
    _next_id = 1

    def __init__(self, cx, cy, frame_idx, crop=None, bbox=None):
        self.id = Track._next_id
        Track._next_id += 1
        self.cx, self.cy = float(cx), float(cy)
        self.first_frame = self.last_frame = frame_idx
        self.history = [(frame_idx, float(cx), float(cy))]
        self.crop = crop  # Save the image crop of the object
        self.bbox = bbox  # Save initial (x1, y1, x2, y2) position

    def update(self, cx, cy, frame_idx):
        self.cx, self.cy = float(cx), float(cy)
        self.last_frame = frame_idx
        self.history.append((frame_idx, float(cx), float(cy)))


def reset_track_ids():
    Track._next_id = 1


def update_tracks(tracks, detections, frame_idx, frame, max_dist=120.0, max_missed=30):
    """Match detections to tracks using nearest Euclidean distance."""
    det_centers = [(float(d[4]), float(d[5])) for d in detections]
    n_det, n_trk = len(det_centers), len(tracks)
    matched_det = [False] * n_det
    matched_trk = [False] * n_trk
    assignments = []

    if n_det and n_trk:
        pairs = []
        for di, (dx, dy) in enumerate(det_centers):
            for ti, trk in enumerate(tracks):
                d = np.hypot(dx - trk.cx, dy - trk.cy)
                if d <= max_dist:
                    pairs.append((d, di, ti))
        pairs.sort(key=lambda p: p[0])

        for d, di, ti in pairs:
            if not matched_det[di] and not matched_trk[ti]:
                matched_det[di] = matched_trk[ti] = True
                assignments.append((di, ti))

    for di, ti in assignments:
        tracks[ti].update(det_centers[di][0], det_centers[di][1], frame_idx)

    # Create new tracks for unmatched detections
    for di in range(n_det):
        if not matched_det[di]:
            x, y, w, h, cx, cy = detections[di]
            x1, y1 = max(0, int(x)), max(0, int(y))
            x2, y2 = min(frame.shape[1], int(x + w)), min(frame.shape[0], int(y + h))
            crop = frame[y1:y2, x1:x2].copy()
            tracks.append(Track(cx, cy, frame_idx, crop=crop, bbox=(x1, y1, x2, y2)))

    tracks = [t for t in tracks if (frame_idx - t.last_frame) <= max_missed]
    return tracks, assignments


def boxes_overlap(box1, box2):
    """Return True if two (x1, y1, x2, y2) bounding boxes intersect."""
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2
    return not (ax2 <= bx1 or ax1 >= bx2 or ay2 <= by1 or ay1 >= by2)


# ==============================================================================
# VERSION 1: GRID SYNOPSIS (Original Code)
# ==============================================================================
def generate_grid_synopsis(valid_tracks, bg_frame, fps, out_dir='Results/Synopsis_Grid', video_name='tabletennis_synopsis_grid.mp4'):
    os.makedirs(out_dir, exist_ok=True)
    grid_cols = 4
    grid_rows = 3
    canvas_h, canvas_w = bg_frame.shape[:2]
    cell_w = canvas_w // grid_cols
    cell_h = canvas_h // grid_rows

    canvas = cv2.cvtColor(bg_frame, cv2.COLOR_GRAY2BGR)
    count_in_canvas = 0
    synopsis_frame_idx = 0

    for t in valid_tracks:
        if t.crop is None or t.crop.size == 0:
            continue

        if count_in_canvas >= (grid_cols * grid_rows):
            synopsis_frame_idx += 1
            cv2.imwrite(f"{out_dir}/synopsis_{synopsis_frame_idx:04d}.png", canvas)
            canvas = cv2.cvtColor(bg_frame, cv2.COLOR_GRAY2BGR)
            count_in_canvas = 0

        row = count_in_canvas // grid_cols
        col = count_in_canvas % grid_cols

        max_w = int(cell_w * 0.8)
        max_h = int(cell_h * 0.8)
        crop_h, crop_w = t.crop.shape[:2]
        scale = min(max_w / crop_w, max_h / crop_h)
        new_w = max(1, int(crop_w * scale))
        new_h = max(1, int(crop_h * scale))
        crop_resized = cv2.resize(t.crop, (new_w, new_h))

        crop_resized_bgr = cv2.cvtColor(crop_resized, cv2.COLOR_GRAY2BGR) if len(crop_resized.shape) == 2 else crop_resized
        ch, cw = crop_resized_bgr.shape[:2]

        start_x = col * cell_w + (cell_w - cw) // 2
        start_y = row * cell_h + (cell_h - ch) // 2 + 20

        canvas[start_y:start_y + ch, start_x:start_x + cw] = crop_resized_bgr

        timestamp = t.first_frame / fps
        text = f"{timestamp:.2f}s"
        cv2.putText(canvas, text, (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        count_in_canvas += 1

    if count_in_canvas > 0:
        synopsis_frame_idx += 1
        cv2.imwrite(f"{out_dir}/synopsis_{synopsis_frame_idx:04d}.png", canvas)

    print(f"[Grid] Saved {synopsis_frame_idx} synopsis frames to {out_dir}/")
    writeVideo(video_name, f"{out_dir}/synopsis_%04d.png", fps=fps)
    print(f"[Grid] Video saved as {video_name}\n")


# ==============================================================================
# VERSION 2: ORIGINAL LOCATION SYNOPSIS (Spatial Code)
# ==============================================================================
def generate_spatial_synopsis(valid_tracks, bg_frame, fps, out_dir='Results/Synopsis_Spatial', video_name='tabletennis_synopsis_spatial.mp4'):
    os.makedirs(out_dir, exist_ok=True)
    canvas = cv2.cvtColor(bg_frame, cv2.COLOR_GRAY2BGR)
    placed_boxes = []
    synopsis_frame_idx = 0

    for t in valid_tracks:
        if t.crop is None or t.crop.size == 0 or t.bbox is None:
            continue

        x1, y1, x2, y2 = t.bbox

        if any(boxes_overlap(t.bbox, existing_box) for existing_box in placed_boxes):
            synopsis_frame_idx += 1
            cv2.imwrite(f"{out_dir}/synopsis_{synopsis_frame_idx:04d}.png", canvas)
            canvas = cv2.cvtColor(bg_frame, cv2.COLOR_GRAY2BGR)
            placed_boxes = []

        crop_bgr = cv2.cvtColor(t.crop, cv2.COLOR_GRAY2BGR) if len(t.crop.shape) == 2 else t.crop
        ch, cw = crop_bgr.shape[:2]

        canvas[y1:y1 + ch, x1:x1 + cw] = crop_bgr
        placed_boxes.append(t.bbox)

        timestamp = t.first_frame / fps
        text = f"{timestamp:.2f}s"
        text_y = y1 - 10 if y1 >= 25 else y1 + 20
        cv2.putText(canvas, text, (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    if len(placed_boxes) > 0:
        synopsis_frame_idx += 1
        cv2.imwrite(f"{out_dir}/synopsis_{synopsis_frame_idx:04d}.png", canvas)

    print(f"[Spatial] Saved {synopsis_frame_idx} synopsis frames to {out_dir}/")
    writeVideo(video_name, f"{out_dir}/synopsis_%04d.png", fps=fps)
    print(f"[Spatial] Video saved as {video_name}\n")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
# --- 1. Run the tracker ---
print("\nStarting Euclidean tracker...")
reset_track_ids()
tracks = []
all_tracks = {}

for i, frame in enumerate(frames1):
    dets, _ = detect_objects(frame, BG0_1, min_area=1500)
    tracks, _ = update_tracks(tracks, dets, i, frame, max_dist=120.0, max_missed=30)
    for t in tracks:
        all_tracks[t.id] = t

print(f"Tracking complete. Created {Track._next_id - 1} unique tracks.")

# --- 2. Prepare valid tracks and FPS ---
valid_tracks = [t for t in all_tracks.values() if len(t.history) >= 10]
valid_tracks.sort(key=lambda t: t.first_frame)
print(f"Generating synopsis for {len(valid_tracks)} valid tracks...")

cap = cv2.VideoCapture('tabletennis_short.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"Original video FPS: {fps}\n")

# --- 3. Run both versions ---
generate_grid_synopsis(valid_tracks, BG0_1, fps)
generate_spatial_synopsis(valid_tracks, BG0_1, fps)

frame size: 1920x1080
fps:30.0
frame count:1330.0
looping thru frames to extract:
extracting frame number 0
extracting frame number 1
extracting frame number 2
extracting frame number 3
extracting frame number 4
extracting frame number 5
extracting frame number 6
extracting frame number 7
extracting frame number 8
extracting frame number 9
extracting frame number 10
extracting frame number 11
extracting frame number 12
extracting frame number 13
extracting frame number 14
extracting frame number 15
extracting frame number 16
extracting frame number 17
extracting frame number 18
extracting frame number 19
extracting frame number 20
extracting frame number 21
extracting frame number 22
extracting frame number 23
extracting frame number 24
extracting frame number 25
extracting frame number 26
extracting frame number 27
extracting frame number 28
extracting frame number 29
extracting frame number 30
extracting frame number 31
extracting frame number 32
extracting frame number 33
extracting

## Task 2: Extend Pipeline with Video Object Segmentation with Memory (SAM2)

In [4]:
#Task 2
#Task requirement: Extend the Task 1 pipeline with a deep-learning model from our research topic
#                  "Video Object Segmentation with Memory". The model must improve one stage of the
#                  pipeline (here: synopsis generation) and the result is saved as an .mp4 video.
#Task overview:
#   1. Load the pretrained SAM 2 video predictor (Ravi et al., 2024, github.com/facebookresearch/sam2).
#   2. Use every valid Task 1 track as a prompt: the bounding box of the object at its first
#      appearance is given to SAM 2 as a box prompt (semi-supervised VOS: only frame 0 is annotated).
#   3. SAM 2 propagates the object through the next frames using its memory bank, giving a
#      pixel-accurate mask of the object in every frame instead of the rectangular Task 1 crop.
#   4. Compare the SAM 2 mask with the Task 1 background-subtraction blob of the same object.
#   5. Keep the frames where the object is clearly visible and skip fragments cut by the image border.
#   6. Build the synopsis again on a colour reference frame: every object is pasted as a
#      segmented cut-out (only the object's pixels, no background box) and it moves inside its
#      cell, objects are added one by one and a new reference frame starts when the grid is full.
#   7. Save all synopsis frames as .png files and create the output video.
#
# Requirements: pip install torch torchvision
#               pip install "git+https://github.com/facebookresearch/sam2.git"
# This cell uses the variables and functions created in the Task 1 cell
# (frames1, BG0_1, valid_tracks, fps, detect_objects, get_BG0, writeVideo, grid_cols, grid_rows, ...),
# so the Task 1 cell must be run first.

import shutil
import urllib.request
import torch
from sam2.build_sam import build_sam2_video_predictor

# ---------------- settings ----------------
video_path = 'tabletennis_short.mp4'
clip_len = 60              # number of frames SAM 2 follows each object (60 frames = 2 seconds)
clear_ratio = 0.5          # a frame is "clear" if the mask is at least 50% of the object's largest mask
min_object_area = 10000    # objects whose largest mask is smaller than this (pixels) are border fragments
add_every = 15             # a new object is added to the synopsis every 15 frames
clip_dir = 'Results/Task2/clip_frames'         # temporary JPEG frames given to SAM 2
synopsis_dir = 'Results/Task2/Synopsis'         # output synopsis frames (.png)
compare_dir = 'Results/Task2/Comparison'        # Task 1 blob vs SAM 2 mask images
os.makedirs(synopsis_dir, exist_ok=True)
os.makedirs(compare_dir, exist_ok=True)

# ---------------- 1. load the pretrained SAM 2 model ----------------
# SAM 2.1 Hiera-Small checkpoint from Meta (downloaded once)
checkpoint = 'sam2.1_hiera_small.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_s.yaml'
if not os.path.exists(checkpoint):
    print("Downloading SAM 2 checkpoint...")
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt', checkpoint)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
predictor = build_sam2_video_predictor(model_cfg, checkpoint, device=device)
print(f"SAM 2 loaded on {device}")

# ---------------- helper functions ----------------
def read_color_frames(video_path, start, count):
    '''reads `count` colour (BGR) frames from the video starting at frame number `start`'''
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(count):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

def first_box_of_track(track):
    '''finds the Task 1 bounding box (x, y, w, h) of a track at the frame where it first appeared'''
    first_idx, cx, cy = track.history[0]
    dets, _ = detect_objects(frames1[first_idx], BG0_1, min_area=1500)
    # the detection whose centre is closest to the track's first centre is the one that started the track
    best = min(dets, key=lambda d: np.hypot(d[4] - cx, d[5] - cy))
    x, y, w, h, _, _ = best
    return x, y, w, h

def segment_track(track, box):
    '''runs SAM 2 on a short clip that starts at the track's first frame, prompted with the
    Task 1 box (x, y, w, h). Returns the colour frames of the clip and one binary mask per frame'''
    start = track.first_frame
    length = min(clip_len, track.last_frame - start + 1)
    clip = read_color_frames(video_path, start, length)

    # SAM 2 reads a video as a folder of JPEG frames named 00000.jpg, 00001.jpg, ...
    shutil.rmtree(clip_dir, ignore_errors=True)
    os.makedirs(clip_dir)
    for i, frame in enumerate(clip):
        cv2.imwrite(f"{clip_dir}/{i:05d}.jpg", frame)

    # box prompt on frame 0 of the clip (the only annotated frame)
    x, y, w, h = box
    sam_box = np.array([x, y, x + w, y + h], dtype=np.float32)   # SAM 2 expects (x1, y1, x2, y2)

    masks = [None] * len(clip)
    with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16, enabled=(device == 'cuda')):
        state = predictor.init_state(video_path=clip_dir, offload_video_to_cpu=True)
        predictor.add_new_points_or_box(state, frame_idx=0, obj_id=track.id, box=sam_box)
        # propagation: every new frame is segmented by reading from the memory bank
        # (the prompted frame + the most recent frames), so the object is followed through the clip
        for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
            masks[frame_idx] = (mask_logits[0, 0] > 0).cpu().numpy()
        predictor.reset_state(state)
    return clip, masks

def mask_iou(mask_a, mask_b):
    '''intersection over union of two binary masks'''
    union = np.logical_or(mask_a, mask_b).sum()
    if union == 0:
        return 0.0
    return np.logical_and(mask_a, mask_b).sum() / union

# ---------------- 2 to 5. segment every valid Task 1 track with SAM 2 ----------------
print(f"\nSegmenting {len(valid_tracks)} tracks with SAM 2...")
segmented = []   # one entry per object: (track, cut-out frames, cut-out masks)
ious = []
for t in valid_tracks:
    bx, by, bw, bh = first_box_of_track(t)
    clip, masks = segment_track(t, (bx, by, bw, bh))

    # ----- 4. compare with the Task 1 blob on the prompted (first) frame -----
    _, blob_mask = detect_objects(frames1[t.first_frame], BG0_1, min_area=1500)
    blob = np.zeros_like(masks[0])
    blob[by:by+bh, bx:bx+bw] = blob_mask[by:by+bh, bx:bx+bw] > 0   # Task 1 blob inside its box
    iou = mask_iou(blob, masks[0])
    ious.append(iou)

    task1_crop = clip[0][by:by+bh, bx:bx+bw]                        # Task 1: rectangular crop
    task1_blob = cv2.cvtColor(blob[by:by+bh, bx:bx+bw].astype(np.uint8) * 255, cv2.COLOR_GRAY2BGR)
    sam2_cut = np.where(masks[0][by:by+bh, bx:bx+bw, None], task1_crop, 0)   # SAM 2: object pixels only
    cv2.imwrite(f"{compare_dir}/track_{t.id:03d}.png", np.hstack([task1_crop, task1_blob, sam2_cut]))
    print(f"track {t.id}: frames {t.first_frame}-{t.first_frame + len(clip) - 1}, IoU(Task 1 blob, SAM 2) = {iou:.2f}")

    # ----- 5. keep the clear frames, skip border fragments -----
    areas = np.array([mask.sum() for mask in masks])   # object size (pixels) in every frame
    if areas.max() < min_object_area:
        print(f"   skipped: largest mask is only {areas.max()} pixels (too small: fragment at the image border or noise)")
        continue
    # SAM 2 keeps following the object while it walks into or out of the view, so keep only
    # the frames where it is clearly visible (at least clear_ratio of its largest size)
    clear = [i for i in range(len(masks)) if areas[i] >= clear_ratio * areas.max()]
    clip = [clip[i] for i in clear]
    masks = [masks[i] for i in clear]

# union box of the object over the clear frames, so every cut-out has the same size
    # and the object can move naturally inside it
    ys, xs = np.nonzero(np.any(masks, axis=0))
    x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    cutouts = [frame[y1:y2, x1:x2].copy() for frame in clip]
    cut_masks = [mask[y1:y2, x1:x2].copy() for mask in masks]
    segmented.append((t, cutouts, cut_masks, (x1, y1, x2, y2)))  # <-- Save the union box (x1, y1, x2, y2) for spatial synopsis

print(f"Mean IoU between Task 1 blobs and SAM 2 masks = {np.mean(ious):.2f}")
print(f"{len(segmented)} of {len(valid_tracks)} objects go into the synopsis")

# ---------------- 6. build the synopsis with segmented cut-outs ----------------
# colour reference frame: same median background as Task 1 (get_BG0), but on colour frames
cap = cv2.VideoCapture(video_path)
sampled = []
for i in range(len(frames1)):
    ret, frame = cap.read()
    if not ret:
        break
    if i % 10 == 0:  # keep every 10th frame to save memory
        sampled.append(frame)
cap.release()
reference = get_BG0(sampled, step=1)
cv2.imwrite('Results/Task2/reference_frame.png', reference)


def boxes_overlap(box1, box2):
    """Return True if two (x1, y1, x2, y2) union boxes intersect."""
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2
    return not (ax2 <= bx1 or ax1 >= bx2 or ay2 <= by1 or ay1 >= by2)


# === VERSION A: GRID SYNOPSIS (Existing Code) ===
grid_synopsis_dir = f"{synopsis_dir}_grid"
os.makedirs(grid_synopsis_dir, exist_ok=True)
grid_cols = 4
grid_rows = 3
canvas_h, canvas_w = BG0_1.shape[:2]
cell_w = canvas_w // grid_cols
cell_h = canvas_h // grid_rows
objects_per_canvas = grid_cols * grid_rows
out_idx = 0
for c in range(0, len(segmented), objects_per_canvas):
    canvas_objects = segmented[c:c + objects_per_canvas]  # objects shown on this reference frame
    longest = max(len(cutouts) for _, cutouts, _, _ in canvas_objects)
    n_frames = (len(canvas_objects) - 1) * add_every + longest

    for k in range(n_frames):
        canvas = reference.copy()
        for slot, (t, cutouts, cut_masks, _) in enumerate(canvas_objects):
            if k < slot * add_every:  # objects are added one by one
                continue
            # the object loops through its segmented frames while it is on the canvas
            j = (k - slot * add_every) % len(cutouts)

            # scale the cut-out to fit in 80% of its grid cell, keeping the aspect ratio
            h, w = cut_masks[0].shape
            scale = min(cell_w * 0.8 / w, cell_h * 0.8 / h)
            new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
            cut = cv2.resize(cutouts[j], (new_w, new_h))
            msk = cv2.resize(cut_masks[j].astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST) > 0

            # centre of the grid cell (shifted down to make room for the timestamp)
            row, col = slot // grid_cols, slot % grid_cols
            start_x = col * cell_w + (cell_w - new_w) // 2
            start_y = row * cell_h + (cell_h - new_h) // 2 + 20

            # paste only the object's pixels (mask), the reference frame stays visible around it
            region = canvas[start_y:start_y + new_h, start_x:start_x + new_w]
            region[msk] = cut[msk]

            # timestamp of the first appearance in the original video above the object
            text = f"{t.first_frame / fps:.2f}s"
            cv2.putText(canvas, text, (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 5)
            cv2.putText(canvas, text, (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        cv2.imwrite(f"{grid_synopsis_dir}/synopsis_{out_idx:04d}.png", canvas)
        out_idx += 1

print(f"Saved {out_idx} grid synopsis frames to {grid_synopsis_dir}")


# === VERSION B: SPATIAL / ORIGINAL LOCATION SYNOPSIS ===
spatial_synopsis_dir = f"{synopsis_dir}_spatial"
os.makedirs(spatial_synopsis_dir, exist_ok=True)

# Group objects into batches whose trajectory union boxes do not overlap
spatial_batches = []
current_batch = []
placed_boxes = []

for item in segmented:
    _, _, _, union_box = item
    if any(boxes_overlap(union_box, existing_box) for existing_box in placed_boxes):
        spatial_batches.append(current_batch)
        current_batch = []
        placed_boxes = []
    current_batch.append(item)
    placed_boxes.append(union_box)

if current_batch:
    spatial_batches.append(current_batch)

out_idx_spatial = 0
for canvas_objects in spatial_batches:
    longest = max(len(cutouts) for _, cutouts, _, _ in canvas_objects)
    n_frames = (len(canvas_objects) - 1) * add_every + longest

    for k in range(n_frames):
        canvas = reference.copy()
        for slot, (t, cutouts, cut_masks, (x1, y1, x2, y2)) in enumerate(canvas_objects):
            if k < slot * add_every:  # objects are added one by one
                continue
            # the object loops through its segmented frames while it is on the canvas
            j = (k - slot * add_every) % len(cutouts)

            cut = cutouts[j]
            msk = cut_masks[j]

            # paste the segmented pixels directly into their original union box location
            region = canvas[y1:y2, x1:x2]
            region[msk] = cut[msk]

            # track the top-left of the object's active mask in frame j so the timestamp moves with it
            ys_j, xs_j = np.nonzero(msk)
            if len(xs_j) > 0:
                label_x = x1 + int(xs_j.min())
                label_y = y1 + int(ys_j.min())
            else:
                label_x, label_y = x1, y1

            text = f"{t.first_frame / fps:.2f}s"
            text_y = label_y - 10 if label_y >= 30 else label_y + 25
            cv2.putText(canvas, text, (label_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 5)
            cv2.putText(canvas, text, (label_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        cv2.imwrite(f"{spatial_synopsis_dir}/synopsis_{out_idx_spatial:04d}.png", canvas)
        out_idx_spatial += 1

shutil.rmtree(clip_dir, ignore_errors=True)
print(f"Saved {out_idx_spatial} spatial synopsis frames to {spatial_synopsis_dir}")

# ---------------- 7. create the output videos ----------------
writeVideo('tabletennis_synopsis_sam2_grid.mp4', f"{grid_synopsis_dir}/synopsis_%04d.png", fps=fps)
print("Task 2 grid synopsis video saved as tabletennis_synopsis_sam2_grid.mp4")

writeVideo('tabletennis_synopsis_sam2_spatial.mp4', f"{spatial_synopsis_dir}/synopsis_%04d.png", fps=fps)
print("Task 2 spatial synopsis video saved as tabletennis_synopsis_sam2_spatial.mp4")

SAM 2 loaded on cpu

Segmenting 16 tracks with SAM 2...


propagate in video: 100%|██████████| 60/60 [01:38<00:00,  1.63s/it]


track 1: frames 0-59, IoU(Task 1 blob, SAM 2) = 0.76


propagate in video: 100%|██████████| 60/60 [01:39<00:00,  1.67s/it]


track 2: frames 41-100, IoU(Task 1 blob, SAM 2) = 0.72


propagate in video: 100%|██████████| 60/60 [01:28<00:00,  1.48s/it]


... (23 outputs left out)

propagate in video: 100%|██████████| 54/54 [01:19<00:00,  1.47s/it]


track 30: frames 1043-1096, IoU(Task 1 blob, SAM 2) = 0.57
   skipped: largest mask is only 1670 pixels (too small: fragment at the image border or noise)


propagate in video: 100%|██████████| 16/16 [00:21<00:00,  1.34s/it]


track 32: frames 1146-1161, IoU(Task 1 blob, SAM 2) = 0.68
   skipped: largest mask is only 1799 pixels (too small: fragment at the image border or noise)
Mean IoU between Task 1 blobs and SAM 2 masks = 0.67
5 of 16 objects go into the synopsis
Saved 120 grid synopsis frames to Results/Task2/Synopsis_grid
Saved 242 spatial synopsis frames to Results/Task2/Synopsis_spatial
frame size: 1920x1080
fps:30.0
frame count:0
looping through image sequence to write vid
writing frame number 0
writing frame number 1
writing frame number 2
writing frame number 3
writing frame number 4
writing frame number 5
writing frame number 6
writing frame number 7
writing frame number 8
writing frame number 9
writing frame number 10
writing frame number 11
writing frame number 12
writing frame number 13
writing frame number 14
writing frame number 15
writing frame number 16
writing frame number 17
writing frame number 18
writing frame number 19
writing frame number 20
writing frame number 21
writing frame numb